In [4]:
#import necessary libraries
import pandas as pd
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [9]:
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [33]:
train_df=pd.read_csv('train.csv')

In [ ]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159571 entries, 0 to 159570
Data columns (total 8 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   id             159571 non-null  object
 1   comment_text   159571 non-null  object
 2   toxic          159571 non-null  int64 
 3   severe_toxic   159571 non-null  int64 
 4   obscene        159571 non-null  int64 
 5   threat         159571 non-null  int64 
 6   insult         159571 non-null  int64 
 7   identity_hate  159571 non-null  int64 
dtypes: int64(6), object(2)
memory usage: 9.7+ MB


In [ ]:
train_df

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0
...,...,...,...,...,...,...,...,...
159566,ffe987279560d7ff,""":::::And for the second time of asking, when ...",0,0,0,0,0,0
159567,ffea4adeee384e90,You should be ashamed of yourself \n\nThat is ...,0,0,0,0,0,0
159568,ffee36eab5c267c9,"Spitzer \n\nUmm, theres no actual article for ...",0,0,0,0,0,0
159569,fff125370e4aaaf3,And it looks like it was actually you who put ...,0,0,0,0,0,0


In [34]:
train_df.comment_text.values[0]

"Explanation\nWhy the edits made under my username Hardcore Metallica Fan were reverted? They weren't vandalisms, just closure on some GAs after I voted at New York Dolls FAC. And please don't remove the template from the talk page since I'm retired now.89.205.38.27"

In [35]:
target_cols = ['toxic','severe_toxic','obscene','threat','insult','identity_hate'] #list of target columns

In [36]:
# looking at the distribution of target columns
for col in target_cols:
    print(train_df[col].value_counts(normalize=True))

toxic
0    0.904156
1    0.095844
Name: proportion, dtype: float64
severe_toxic
0    0.990004
1    0.009996
Name: proportion, dtype: float64
obscene
0    0.947052
1    0.052948
Name: proportion, dtype: float64
threat
0    0.997004
1    0.002996
Name: proportion, dtype: float64
insult
0    0.950636
1    0.049364
Name: proportion, dtype: float64
identity_hate
0    0.991195
1    0.008805
Name: proportion, dtype: float64


In [ ]:
# Text preprocessing function
def preprocess_text(text):
    # Remove special characters and punctuation
    text = re.sub(r'[^\w\s]', '', text)
    # Convert to lowercase
    text = text.lower()
    # Tokenize text
    tokens = word_tokenize(text)
    # Remove stopwords
    tokens = [word for word in tokens if word not in stopwords.words('english')]
    # Join tokens back into a string
    processed_text = ' '.join(tokens)
    return processed_text

# Apply preprocessing to the 'text' column
train_df['clean_text'] = train_df['comment_text'].apply(preprocess_text)

In [10]:
from torchtext.data.utils import get_tokenizer

In [ ]:
!pip install torchtext

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 29.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 779.1/779.1 MB 6.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 15.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 48.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 47.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 10.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 32.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 47.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 31.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 28.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━

In [11]:
tokenizer = get_tokenizer('basic_english')

In [37]:
sample_text = train_df.comment_text.values[1]
sample_text

"D'aww! He matches this background colour I'm seemingly stuck with. Thanks.  (talk) 21:51, January 11, 2016 (UTC)"

In [ ]:
sample_tokens = tokenizer(sample_text)[:20] # lower case , individual words
sample_tokens

['d',
 "'",
 'aww',
 '!',
 'he',
 'matches',
 'this',
 'background',
 'colour',
 'i',
 "'",
 'm',
 'seemingly',
 'stuck',
 'with',
 '.',
 'thanks',
 '.',
 '(',
 'talk']

In [38]:
from torchtext.vocab import build_vocab_from_iterator

In [39]:
VOCAB_SIZE = 1500
unknown_token = '<unk>'
pad_token = '<pad>'

In [40]:
comment_tokens = train_df.comment_text.map(tokenizer)

In [41]:
vocab = build_vocab_from_iterator(comment_tokens,
                                  specials = [unknown_token,pad_token],
                                  max_tokens = VOCAB_SIZE) # assigns a number to tokens

In [42]:
vocab['it']

14

In [43]:
vocab[unknown_token] , vocab[pad_token]

(0, 1)

In [44]:
vocab.set_default_index(vocab[unknown_token])

In [45]:
vocab['background']

1443

In [ ]:
!pip install matplotlib

In [13]:
# function to truncate sentences excedding max_length or pad when length is less than max_length

def pad_tokens(tokens):
    if len(tokens) > MAX_LENGTH:
        return tokens[:MAX_LENGTH]
    else:
        return tokens + [pad_token] * (MAX_LENGTH - len(tokens))

In [14]:
# example of the functions
MAX_LENGTH = 5
# truncation
test1 = 'I am doing good, how are you?'
print(pad_tokens(tokenizer(test1)))

['i', 'am', 'doing', 'good', ',']


In [15]:
# padding
test1 = "I'm good"
print(pad_tokens(tokenizer(test1)))

['i', "'", 'm', 'good', '<pad>']


In [16]:
MAX_LENGTH = 150

In [17]:
import torch
from torch.utils.data import Dataset

In [ ]:
train_df.columns

Index(['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat',
       'insult', 'identity_hate', 'clean_text'],
      dtype='object')

In [18]:
class Jigsaw(Dataset):
    def __init__(self , df , test_dataset=False):
        super().__init__()
        self.df = df
        self.test_dataset = test_dataset

    def __getitem__(self,index):
        comment_text = self.df.comment_text.values[index]
        comment_tokens = pad_tokens(tokenizer(comment_text))
        input = torch.tensor(vocab.lookup_indices(comment_tokens))
        if self.test_dataset:
            target = torch.tensor([0,0,0,0,0,0]).float() # for test dataset make the target as all zeros , which is not in use
        else:
            target = torch.tensor(self.df[target_cols].values[index]).float()
        return input , target

    def __len__(self):
        return len(self.df)


In [ ]:
train_dataset = Jigsaw(train_df)

In [ ]:
train_dataset[2]

(tensor([406, 439,   4,   6,   9,  81, 145,  19, 257,   5,  86, 330,   2,  14,
           9,  28,  61,  13,  18, 594,  12,   0, 486, 496, 111,   8, 596,   5,
          46, 328, 140, 360,   7,  39,  50,  38,   2,  59, 212,   5, 426,  69,
          47,   3,   0, 102,   3, 704, 482,   2,   1,   1,   1,   1,   1,   1,
           1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,
           1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,
           1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,
           1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,
           1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,
           1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,
           1,   1,   1,   1,   1,   1,   1,   1,   1,   1]),
 tensor([0., 0., 0., 0., 0., 0.]))

In [ ]:
from torch.utils.data import random_split

In [ ]:
validation_fraction = 0.25

In [ ]:
train_ds , val_ds = random_split(train_dataset, [1-validation_fraction, validation_fraction])

In [61]:
test_df=pd.read_csv('test.csv')

In [62]:
test_df

,id,comment_text
0,00001cee341fdb12,Yo bitch Ja Rule is more succesful then you'll...
1,0000247867823ef7,== From RfC == \n\n The title is fine as it is...
2,00013b17ad220c46,""" \n\n == Sources == \n\n * Zawe Ashton on Lap..."
3,00017563c3f7919a,":If you have a look back at the source, the in..."
4,00017695ad8997eb,I don't anonymously edit articles at all.
...,...,...
153159,fffcd0960ee309b5,". \n i totally agree, this stuff is nothing bu..."
153160,fffd7a9a6eb32c16,== Throw from out field to home plate. == \n\n...
153161,fffda9e8d6fafa9e,""" \n\n == Okinotorishima categories == \n\n I ..."
153162,fffe8f1340a79fc2,""" \n\n == """"One of the founding nations of the..."


In [63]:
test_dataset = Jigsaw(test_df,test_dataset=True)

In [22]:
from torch.utils.data import DataLoader

In [ ]:
BATCH_SIZE = 128

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE*2, num_workers=4, pin_memory=True)
test_dl = DataLoader(test_dataset, batch_size=BATCH_SIZE*2, num_workers=4, pin_memory=True)

In [ ]:
for a_input , a_target in train_dl:
    print(a_input.shape)
    print(a_target.shape)
    break

torch.Size([128, 150])
torch.Size([128, 6])


In [27]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

In [28]:
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, embedding_dim, padding_idx=1)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.fc1 = nn.Linear(hidden_dim, 6)
        self.learning_rate = 0.01
        self.criterion = nn.BCEWithLogitsLoss()

    def forward(self, x):
        out = self.emb(x)
        out, _ = self.lstm(out)
        out = F.relu(out[:, -1, :])
        out = self.fc1(out)
        return out

    def training_step(self, batch, batch_idx):
        inputs, targets = batch
        outputs = self(inputs)
        loss = self.criterion(outputs, targets)
        return loss

    def validation_step(self, batch, batch_idx):
        inputs, targets = batch
        outputs = self(inputs)
        loss = self.criterion(outputs, targets)
        return loss.item()

    def validation_epoch_end(self, validation_step_outputs):
        loss = np.mean(validation_step_outputs)
        print("Epoch #{}; Loss: {:4f} ".format(self.current_epoch, loss))

    def predict_step(self, batch, batch_idx):
        inputs, _ = batch
        outputs = self(inputs)
        probs = torch.sigmoid(outputs)
        return probs

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.learning_rate)

In [29]:
model = LSTMModel(vocab_size=VOCAB_SIZE, embedding_dim=256, hidden_dim=128)

In [ ]:
for batch in train_dl:
    b_inputs, b_targets = batch
    print('b_input.shape', b_inputs.shape)
    print('b_targets.shape', b_targets.shape)

    outputs = model(b_inputs)
    print('outputs.shape', outputs.shape)

    # Calculate probabilities using sigmoid activation
    probs = torch.sigmoid(outputs)

    # Calculate the loss using BCEWithLogitsLoss
    loss = model.criterion(outputs, b_targets)
    print('Loss', loss.item())
    break


b_input.shape torch.Size([128, 150])
b_targets.shape torch.Size([128, 6])
outputs.shape torch.Size([128, 6])
Loss 0.6864359974861145


In [47]:
# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


LSTMModel(
  (emb): Embedding(1500, 256, padding_idx=1)
  (lstm): LSTM(256, 128, batch_first=True)
  (fc1): Linear(in_features=128, out_features=6, bias=True)
  (criterion): BCEWithLogitsLoss()
)

In [ ]:
# Define the optimizer (using a dummy learning rate for now)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
# Train your model using the defined optimizer and criterion for 3 epochs
for epoch in range(10):
    model.train()  # Set model to training mode
    for batch in train_dl:
        # Move data to GPU if available
        inputs, targets = batch[0].to(device), batch[1].to(device)

        # Forward pass
        outputs = model(inputs)

        # Calculate probabilities using sigmoid activation
        probs = torch.sigmoid(outputs)

        # Calculate the loss using BCEWithLogitsLoss
        loss = model.criterion(outputs, targets)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        print('Epoch [{}/{}], Batch Loss: {:.4f}'.format(epoch+1, 3, loss.item()))

    # Validate your model after each epoch
    model.eval()  # Set model to evaluation mode
    total_loss = 0
    with torch.no_grad():
        for batch in val_dl:
            # Move data to GPU if available
            inputs, targets = batch[0].to(device), batch[1].to(device)

            # Forward pass
            outputs = model(inputs)

            # Calculate probabilities using sigmoid activation
            probs = torch.sigmoid(outputs)

            # Calculate the loss using BCEWithLogitsLoss
            loss = model.criterion(outputs, targets)

            # Accumulate validation loss
            total_loss += loss.item() * inputs.size(0)

    # Calculate average validation loss for the epoch
    avg_val_loss = total_loss / len(val_ds)
    print(f"Epoch {epoch + 1}/{3}, Validation Loss: {avg_val_loss:.4f}")

Epoch [1/3], Batch Loss: 0.6865
Epoch [1/3], Batch Loss: 0.6813
Epoch [1/3], Batch Loss: 0.6774
Epoch [1/3], Batch Loss: 0.6713
Epoch [1/3], Batch Loss: 0.6687
Epoch [1/3], Batch Loss: 0.6624
Epoch [1/3], Batch Loss: 0.6594
Epoch [1/3], Batch Loss: 0.6548
Epoch [1/3], Batch Loss: 0.6501
Epoch [1/3], Batch Loss: 0.6357
Epoch [1/3], Batch Loss: 0.6286
Epoch [1/3], Batch Loss: 0.6296
Epoch [1/3], Batch Loss: 0.6162
Epoch [1/3], Batch Loss: 0.6066
Epoch [1/3], Batch Loss: 0.5907
Epoch [1/3], Batch Loss: 0.5609
Epoch [1/3], Batch Loss: 0.5066
Epoch [1/3], Batch Loss: 0.4648
Epoch [1/3], Batch Loss: 0.4258
Epoch [1/3], Batch Loss: 0.3737
Epoch [1/3], Batch Loss: 0.3450
Epoch [1/3], Batch Loss: 0.3048
Epoch [1/3], Batch Loss: 0.3033
Epoch [1/3], Batch Loss: 0.2752
Epoch [1/3], Batch Loss: 0.2794
Epoch [1/3], Batch Loss: 0.2704
Epoch [1/3], Batch Loss: 0.2342
Epoch [1/3], Batch Loss: 0.2541
Epoch [1/3], Batch Loss: 0.2102
Epoch [1/3], Batch Loss: 0.2252
Epoch [1/3], Batch Loss: 0.2283
Epoch [1

In [1]:
import pickle

In [ ]:
# Save the model to a .pkl file
torch.save(model.state_dict(), 'model.pkl')

In [30]:
# Load the saved model from file
state_dict = torch.load('/content/model.pkl')

# Load the state dict into the model
model.load_state_dict(state_dict)


<All keys matched successfully>

In [ ]:
test_df

,id,comment_text
0,00001cee341fdb12,Yo bitch Ja Rule is more succesful then you'll...
1,0000247867823ef7,== From RfC == \n\n The title is fine as it is...
2,00013b17ad220c46,""" \n\n == Sources == \n\n * Zawe Ashton on Lap..."
3,00017563c3f7919a,":If you have a look back at the source, the in..."
4,00017695ad8997eb,I don't anonymously edit articles at all.
...,...,...
153159,fffcd0960ee309b5,". \n i totally agree, this stuff is nothing bu..."
153160,fffd7a9a6eb32c16,== Throw from out field to home plate. == \n\n...
153161,fffda9e8d6fafa9e,""" \n\n == Okinotorishima categories == \n\n I ..."
153162,fffe8f1340a79fc2,""" \n\n == """"One of the founding nations of the..."


In [55]:
submission_df=pd.read_csv('/content/sample_submission.csv')
submission_df

,id,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,00001cee341fdb12,0.5,0.5,0.5,0.5,0.5,0.5
1,0000247867823ef7,0.5,0.5,0.5,0.5,0.5,0.5
2,00013b17ad220c46,0.5,0.5,0.5,0.5,0.5,0.5
3,00017563c3f7919a,0.5,0.5,0.5,0.5,0.5,0.5
4,00017695ad8997eb,0.5,0.5,0.5,0.5,0.5,0.5
...,...,...,...,...,...,...,...
153159,fffcd0960ee309b5,0.5,0.5,0.5,0.5,0.5,0.5
153160,fffd7a9a6eb32c16,0.5,0.5,0.5,0.5,0.5,0.5
153161,fffda9e8d6fafa9e,0.5,0.5,0.5,0.5,0.5,0.5
153162,fffe8f1340a79fc2,0.5,0.5,0.5,0.5,0.5,0.5


In [65]:
# Make predictions on the test dataset
model.eval()  # Set model to evaluation mode
test_probs = []
with torch.no_grad():
    for batch in test_dl:
        # Move data to GPU if available
        inputs = batch[0].to(device)

        # Forward pass
        outputs = model(inputs)

        # Calculate probabilities using sigmoid activation
        probs = torch.sigmoid(outputs)

        # Append probabilities to the list
        test_probs.append(probs)

# Concatenate probabilities from all batches
test_probs = torch.cat(test_probs)

# Convert probabilities to numpy array and assign to submission dataframe
submission_df[target_cols] = test_probs.detach().cpu().numpy()

# Save submission dataframe to CSV file
submission_df.to_csv('submission.csv', index=None)

In [66]:
submission_result_df=pd.read_csv('submission.csv')
submission_result_df

,id,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,00001cee341fdb12,0.993809,3.217910e-01,0.975901,0.051884,0.870420,0.200248
1,0000247867823ef7,0.001030,2.427304e-06,0.000202,0.000008,0.000090,0.000038
2,00013b17ad220c46,0.000585,2.599704e-06,0.000107,0.000012,0.000051,0.000025
3,00017563c3f7919a,0.000275,8.904606e-07,0.000064,0.000004,0.000027,0.000012
4,00017695ad8997eb,0.000479,1.465778e-06,0.000087,0.000007,0.000035,0.000013
...,...,...,...,...,...,...,...
153159,fffcd0960ee309b5,0.001055,3.176159e-06,0.000156,0.000014,0.000076,0.000027
153160,fffd7a9a6eb32c16,0.002258,5.056394e-06,0.000319,0.000023,0.000216,0.000101
153161,fffda9e8d6fafa9e,0.000368,1.181667e-06,0.000106,0.000005,0.000039,0.000021
153162,fffe8f1340a79fc2,0.003387,4.251925e-06,0.000533,0.000024,0.000280,0.000081


In [5]:
test_labels_df=pd.read_csv('test_labels.csv')

In [6]:
test_labels_df

,id,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,00001cee341fdb12,-1,-1,-1,-1,-1,-1
1,0000247867823ef7,-1,-1,-1,-1,-1,-1
2,00013b17ad220c46,-1,-1,-1,-1,-1,-1
3,00017563c3f7919a,-1,-1,-1,-1,-1,-1
4,00017695ad8997eb,-1,-1,-1,-1,-1,-1
...,...,...,...,...,...,...,...
153159,fffcd0960ee309b5,-1,-1,-1,-1,-1,-1
153160,fffd7a9a6eb32c16,-1,-1,-1,-1,-1,-1
153161,fffda9e8d6fafa9e,-1,-1,-1,-1,-1,-1
153162,fffe8f1340a79fc2,-1,-1,-1,-1,-1,-1


In [7]:
filtered_df = test_labels_df[(test_labels_df.iloc[:, 2:] != -1).all(axis=1)]

In [8]:
filtered_df

,id,toxic,severe_toxic,obscene,threat,insult,identity_hate
5,0001ea8717f6de06,0,0,0,0,0,0
7,000247e83dcc1211,0,0,0,0,0,0
11,0002f87b16116a7f,0,0,0,0,0,0
13,0003e1cccfd5a40a,0,0,0,0,0,0
14,00059ace3e3e9a53,0,0,0,0,0,0
...,...,...,...,...,...,...,...
153150,fff8f64043129fa2,0,0,0,0,0,0
153151,fff9d70fe0722906,0,0,0,0,0,0
153154,fffa8a11c4378854,0,0,0,0,0,0
153155,fffac2a094c8e0e2,1,0,1,0,1,0


In [49]:
# Make predictions on the test dataset
model.eval()  # Set model to evaluation mode
predicted_labels = []
with torch.no_grad():
    for batch in test_dl:
        # Move data to GPU if available
        inputs = batch[0].to(device)

        # Forward pass
        outputs = model(inputs)

        # Convert outputs to binary predictions
        binary_predictions = (outputs > 0.5).int()

        # Append binary predictions to the list
        predicted_labels.append(binary_predictions)

# Concatenate binary predictions from all batches
predicted_labels = torch.cat(predicted_labels)

# Convert binary predictions to numpy array
predicted_labels = predicted_labels.detach().cpu().numpy()

# Combine the true labels and predicted labels for each label into a single DataFrame
result_df = test_df[['comment_text']].copy()  # Copy comment_text column from test_df
for label in ['toxic', 'severe_toxic', 'threat', 'obscene', 'insult', 'identity_hate']:
    # Concatenate true labels and predicted labels for each label
    result_df[label + '_true'] = test_labels_df[label]
    result_df[label + '_predicted'] = predicted_labels[:, target_cols.index(label)]

# Save the result DataFrame to a CSV file
result_df.to_csv('result.csv', index=False)

/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py:558: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


In [50]:
result_df

,comment_text,toxic_true,toxic_predicted,severe_toxic_true,severe_toxic_predicted,threat_true,threat_predicted,obscene_true,obscene_predicted,insult_true,insult_predicted,identity_hate_true,identity_hate_predicted
0,Yo bitch Ja Rule is more succesful then you'll...,-1,1,-1,0,-1,0,-1,1,-1,1,-1,0
1,== From RfC == \n\n The title is fine as it is...,-1,0,-1,0,-1,0,-1,0,-1,0,-1,0
2,""" \n\n == Sources == \n\n * Zawe Ashton on Lap...",-1,0,-1,0,-1,0,-1,0,-1,0,-1,0
3,":If you have a look back at the source, the in...",-1,0,-1,0,-1,0,-1,0,-1,0,-1,0
4,I don't anonymously edit articles at all.,-1,0,-1,0,-1,0,-1,0,-1,0,-1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
142297,". \n i totally agree, this stuff is nothing bu...",0,0,0,0,0,0,0,0,0,0,0,0
142298,== Throw from out field to home plate. == \n\n...,-1,0,-1,0,-1,0,-1,0,-1,0,-1,0
142299,""" \n\n == Okinotorishima categories == \n\n I ...",-1,0,-1,0,-1,0,-1,0,-1,0,-1,0
142300,""" \n\n == """"One of the founding nations of the...",0,0,0,0,0,0,0,0,0,0,0,0


In [52]:
# remove the '-1' values
filtered_df = result_df[(result_df.iloc[:, 1:] != -1).all(axis=1)]

# Print the filtered DataFrame
print(filtered_df)

                                             comment_text  toxic_true  \
5       Thank you for understanding. I think very high...           0   
7                        :Dear god this site is horrible.           0   
11      "::: Somebody will invariably try to add Relig...           0   
13      " \n\n It says it right there that it IS a typ...           0   
14      " \n\n == Before adding a new product to the l...           0   
...                                                   ...         ...   
142287  shut down the mexican border withought looking...           0   
142291  " \n\n ==""Illness"" no shit== \n Just for the...           0   
142294  " \n\n == Unicorn lair discovery == \n\n Suppo...           0   
142297  . \n i totally agree, this stuff is nothing bu...           0   
142300  " \n\n == ""One of the founding nations of the...           0   

        toxic_predicted  severe_toxic_true  severe_toxic_predicted  \
5                     0                  0           

In [54]:
# Write the filtered DataFrame to a CSV file
filtered_df.to_csv('results_gt_pred.csv', index=False)


In [53]:
# Calculate accuracy for each label
accuracies = {}
for label in ['toxic', 'severe_toxic', 'threat', 'obscene', 'insult', 'identity_hate']:
    # Calculate accuracy for the label
    correct_predictions = (filtered_df[label + '_true'] == filtered_df[label + '_predicted']).sum()
    total_predictions = len(filtered_df)
    accuracy = correct_predictions / total_predictions
    accuracies[label] = accuracy

# Print accuracies
for label, accuracy in accuracies.items():
    print("Accuracy for {}: {:.4f}".format(label, accuracy))


Accuracy for toxic: 0.7749
Accuracy for severe_toxic: 0.9934
Accuracy for threat: 0.9968
Accuracy for obscene: 0.8665
Accuracy for insult: 0.8863
Accuracy for identity_hate: 0.9889


In [67]:
# Calculate combined accuracy
combined_accuracy = np.mean(list(accuracies.values()))

# Print accuracies
print("Combined Accuracy: {:.4f}".format(combined_accuracy))

Combined Accuracy: 0.9178
